# Phase 10 — Data Quality & Schema Enforcement Experiments Notebook

This is the **STARTER notebook** for Phase 10.

Run it **top-to-bottom**. The deterministic setup, experiment order, validation questions, and applied-project structure are preserved from the SOLUTION notebook.

Worked implementation and answer-revealing code have been removed so you can build the validation layer yourself.

Use this workflow:

```text
state the contract
    ↓
state the grain
    ↓
classify the rule
    ↓
predict invalid rows
    ↓
implement ONE validation concern
    ↓
preserve rejection evidence
    ↓
split accepted/rejected
    ↓
reconcile counts
    ↓
review Spark execution
```

Core question:

> **Can this validation layer protect downstream transformations without silently deleting bad data or scattering rules throughout the pipeline?**

Important:

- Use **PySpark 4.2.0**.
- Use `from pyspark.sql import functions as F`.
- Use single-quoted Python strings.
- Include concise inline teaching comments.
- Preserve diagnostic rejection reasons.
- State grain before validating keys or relationships.
- Do not use `dropDuplicates()` to hide unexplained key violations.
- Do not mark Phase 10 complete from this notebook alone.


<a id="toc"></a>
## Table of Contents

- [Setup and Practice Data](#setup-and-practice-data)
- [Data-Quality Exercise Protocol](#exercise-protocol)
- [Experiment 1 — Structural Schema Contract](#experiment-1)
- [Experiment 2 — Required Fields](#experiment-2)
- [Experiment 3 — Domain, Range, and Business Rules](#experiment-3)
- [Experiment 4 — Primary-Key Uniqueness](#experiment-4)
- [Experiment 5 — Composite-Key Validation](#experiment-5)
- [Experiment 6 — Exact Duplicates vs. Duplicate Business Keys](#experiment-6)
- [Experiment 7 — Referential Integrity](#experiment-7)
- [Experiment 8 — Multiple Rejection Reasons](#experiment-8)
- [Experiment 9 — Accepted vs. Rejected Records](#experiment-9)
- [Experiment 10 — Quarantine Design](#experiment-10)
- [Experiment 11 — Validation Metrics and Reconciliation](#experiment-11)
- [Experiment 12 — Distributed Cost of Validation](#experiment-12)
- [Applied Phase 10 Project](#applied-project)
- [Cleanup](#cleanup)


<a id="setup-and-practice-data"></a>
# Setup and Practice Data

Declared grains:

```text
orders_df
= one row per order_id

customers_df
= one row per customer_id

inventory_df
= one row per snapshot_date + store_id + product_id
```

The data intentionally contains quality defects so the validation layer has something real to detect.

[Back to Table of Contents](#toc)


In [ ]:
from datetime import date
from decimal import Decimal

from pyspark.sql import Column
from pyspark.sql import DataFrame
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DateType
from pyspark.sql.types import DecimalType
from pyspark.sql.types import IntegerType
from pyspark.sql.types import StringType
from pyspark.sql.types import StructField
from pyspark.sql.types import StructType


spark = (
    SparkSession.builder
    .appName('phase_10_data_quality_experiments')
    .master('local[4]')
    # Keep the teaching workload small and predictable.
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')


In [ ]:
# Incoming schemas allow NULL values so invalid rows can be diagnosed
# instead of failing before the quality layer sees them.
ORDERS_SCHEMA = StructType(
    [
        StructField('order_id', StringType(), nullable=True),
        StructField('customer_id', StringType(), nullable=True),
        StructField('order_date', DateType(), nullable=True),
        StructField('order_status', StringType(), nullable=True),
        StructField('net_sales', DecimalType(12, 2), nullable=True),
    ]
)

CUSTOMERS_SCHEMA = StructType(
    [
        StructField('customer_id', StringType(), nullable=True),
        StructField('customer_name', StringType(), nullable=True),
        StructField('province', StringType(), nullable=True),
        StructField('customer_segment', StringType(), nullable=True),
    ]
)

INVENTORY_SCHEMA = StructType(
    [
        StructField('snapshot_date', DateType(), nullable=True),
        StructField('store_id', StringType(), nullable=True),
        StructField('product_id', StringType(), nullable=True),
        StructField('quantity_on_hand', IntegerType(), nullable=True),
    ]
)

ORDERS_ROWS = [
    ('O001', 'C001', date(2026, 9, 1), 'COMPLETED', Decimal('125.00')),
    (None, 'C002', date(2026, 9, 1), 'COMPLETED', Decimal('80.00')),
    ('O003', 'C001', date(2026, 9, 2), 'UNKNOWN', Decimal('45.00')),
    ('O004', 'C003', date(2026, 9, 2), 'COMPLETED', Decimal('-20.00')),
    ('O005', 'C999', date(2026, 9, 3), 'COMPLETED', Decimal('30.00')),
    ('O006', 'C004', date(2026, 9, 3), 'COMPLETED', Decimal('10.00')),
    ('O006', 'C004', date(2026, 9, 3), 'COMPLETED', Decimal('15.00')),
    ('O007', 'C004', date(2026, 9, 4), 'PENDING', Decimal('20.00')),
    ('O007', 'C004', date(2026, 9, 4), 'PENDING', Decimal('20.00')),
    ('O008', 'C003', date(2026, 9, 4), 'CANCELLED', Decimal('12.00')),
    ('O009', 'C002', date(2026, 9, 5), 'CANCELLED', Decimal('0.00')),
]

CUSTOMERS_ROWS = [
    ('C001', 'Alice Wong', 'ON', 'CONSUMER'),
    ('C002', 'Ben Tremblay', 'QC', 'CONSUMER'),
    ('C003', 'Carla Singh', 'BC', 'BUSINESS'),
    ('C004', 'Diego Martin', 'ON', 'CONSUMER'),
]

INVENTORY_ROWS = [
    (date(2026, 9, 5), 'S001', 'P001', 10),
    (date(2026, 9, 5), 'S001', 'P002', 7),
    (date(2026, 9, 5), 'S002', 'P001', 5),
    (date(2026, 9, 5), 'S002', 'P001', 8),
    (date(2026, 9, 5), 'S003', 'P003', -1),
]

orders_df = spark.createDataFrame(ORDERS_ROWS, schema=ORDERS_SCHEMA)
customers_df = spark.createDataFrame(CUSTOMERS_ROWS, schema=CUSTOMERS_SCHEMA)
inventory_df = spark.createDataFrame(INVENTORY_ROWS, schema=INVENTORY_SCHEMA)

orders_df.orderBy(F.col('order_id').asc_nulls_first()).show(truncate=False)
customers_df.orderBy('customer_id').show(truncate=False)
inventory_df.orderBy('snapshot_date', 'store_id', 'product_id').show(truncate=False)


<a id="exercise-protocol"></a>
# Data-Quality Exercise Protocol

For each validation concern:

```text
1. State the contract.
2. State the grain.
3. Classify the rule:
   row-level / dataset-level / cross-dataset.
4. Predict which rows are invalid.
5. Predict whether Spark needs:
   narrow expressions / shuffle / join / action.
6. Implement the rule.
7. Preserve diagnostic reasons.
8. Verify grain and classification.
9. Reconcile counts.
10. Explain the downstream defect prevented.
```

[Back to Table of Contents](#toc)


<a id="experiment-1"></a>
# Experiment 1 — Structural Schema Contract

Contract:

```text
orders_df must contain the expected columns with the expected Spark data types.
```

This is a **structural** check. It is not a row-level content rule.

Prediction:

```text
correct orders_df
→ pass

missing net_sales
→ fail structural contract

net_sales as string
→ fail structural contract
```

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


<a id="experiment-2"></a>
# Experiment 2 — Required Fields

Declared order grain:

```text
one row per order_id
```

Required-field rules are **row-level** rules.

For required string identifiers, both are invalid:

```text
NULL
''
```

Expected rejection reason for the provided data:

```text
MISSING_ORDER_ID
```

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


<a id="experiment-3"></a>
# Experiment 3 — Domain, Range, and Business Rules

Three different row-level rule types:

```text
DOMAIN
order_status ∈ {COMPLETED, CANCELLED, PENDING}

RANGE
net_sales >= 0.00

CROSS-COLUMN BUSINESS RULE
if order_status = CANCELLED
then net_sales = 0.00
```

Expected failures:

```text
O003 → INVALID_ORDER_STATUS
O004 → NEGATIVE_NET_SALES
O008 → INVALID_CANCELLED_AMOUNT
```

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


<a id="experiment-4"></a>
# Experiment 4 — Primary-Key Uniqueness

Declared grain:

```text
orders_df
= one row per order_id
```

Therefore:

```text
order_id must be present
AND
order_id must be unique
```

Primary-key uniqueness is a **dataset-level** rule.

Expected duplicate keys:

```text
O006
O007
```

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


<a id="experiment-5"></a>
# Experiment 5 — Composite-Key Validation

Declared inventory grain:

```text
one row per snapshot_date + store_id + product_id
```

The individual columns may repeat.

The **combination** must be unique.

Expected duplicate composite key:

```text
2026-09-05 + S002 + P001
```

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


<a id="experiment-6"></a>
# Experiment 6 — Exact Duplicates vs. Duplicate Business Keys

These are different concepts.

```text
O006
→ duplicate business key with conflicting values

O007
→ duplicate business key AND exact duplicate row
```

`dropDuplicates()` should not be used as a generic validation strategy because it hides the defect and may choose a survivor without a defined business rule.

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


<a id="experiment-7"></a>
# Experiment 7 — Referential Integrity

Relationship:

```text
orders.customer_id
    →
customers.customer_id
```

Referential integrity is a **cross-dataset** rule.

Expected orphan:

```text
O005 → customer_id C999
```

Before trusting the relationship, verify that the parent relation itself has the expected grain:

```text
customers_df
= one row per customer_id
```

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


<a id="experiment-8"></a>
# Experiment 8 — Multiple Rejection Reasons

A row may violate more than one rule.

For diagnostics, preserve all meaningful failures rather than collapsing them into:

```text
INVALID_ROW
```

We'll add one synthetic row:

```text
order_id = NULL
customer_id = C999
order_status = UNKNOWN
net_sales = -10.00
```

Expected reasons include:

```text
MISSING_ORDER_ID
INVALID_ORDER_STATUS
NEGATIVE_NET_SALES
ORPHAN_CUSTOMER_ID
```

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


<a id="experiment-9"></a>
# Experiment 9 — Accepted vs. Rejected Records

Split only **after** all required validation rules have been evaluated.

Classification invariant:

```text
Every interpretable input row
→ accepted OR rejected
```

Accepted rows:

```text
size(rejection_reasons) == 0
```

Rejected rows:

```text
size(rejection_reasons) > 0
```

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


<a id="experiment-10"></a>
# Experiment 10 — Quarantine Design

Rejected records should become a first-class output.

Preserve:

```text
original business columns
rejection_reasons
```

Add deterministic metadata:

```text
source_dataset
validation_run_date
```

Do not mutate away the original source defect before quarantine.

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


<a id="experiment-11"></a>
# Experiment 11 — Validation Metrics and Reconciliation

Core metrics:

```text
input_count
accepted_count
rejected_count
acceptance_rate
rejection_rate
rejection count by reason
```

Mandatory reconciliation:

```text
input_count
=
accepted_count + rejected_count
```

Because a rejected row can fail multiple rules:

```text
sum(reason counts)
can exceed
rejected_count
```

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


<a id="experiment-12"></a>
# Experiment 12 — Distributed Cost of Validation

Predict before inspecting:

```text
row-level expressions
→ usually narrow

groupBy() uniqueness checks
→ shuffle / Exchange

referential-integrity joins
→ distributed join strategy

count / first / show / write
→ actions or materialization
```

Correctness comes first. Expensive validation should be optimized with evidence, not deleted because it costs Spark work.

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


<a id="applied-project"></a>
# Applied Phase 10 Project

Build one reusable retail data-quality layer spanning:

```text
orders
customers
inventory
```

Required architecture:

```text
Raw
 ↓
Schema enforcement
 ↓
Reusable validation
 ├── Accepted → transformations
 └── Rejected → quarantine
```

The important design rule is:

```text
validation owns source correctness
transformations own business results
```

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


In [ ]:
# TODO:
# Implement this experiment using the contract, grain, and predictions above.
#
# Requirements:
# - Use single-quoted Python strings.
# - Include concise inline comments explaining WHAT the validation does and WHY.
# - Preserve rejected-row evidence rather than silently filtering it away.
# - Keep the resulting DataFrame grain explicit.
#
# Write your solution below.


## Applied Review

After completing the applied task, review whether your implementation demonstrates the full Phase 10 curriculum:

```text
schema validation
required fields
domain validation
range checks
multi-column business rules
primary-key uniqueness
composite-key uniqueness
duplicate detection
referential integrity
rejection reasons
accepted vs. rejected records
quarantine
validation metrics
reconciliation
validation / transformation separation
```

Do not use the solution notebook until you have attempted the implementation and explained your design choices.

This notebook does **not** perform the formal Phase 10 mastery gate.

[Back to Table of Contents](#toc)


<a id="cleanup"></a>
# Cleanup

Stop the local teaching session when finished.

[Back to Table of Contents](#toc)


In [ ]:
spark.stop()
